In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('cardekho_imputated.csv')

In [3]:
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


# Data Claening

In [4]:
#check null values
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [5]:
#Remove Unnecesary columns
df.drop(columns=['Unnamed: 0','car_name','brand'],axis=1,inplace=True)

In [6]:
# dependent and independent features
X=df.drop('selling_price',axis=1)
y=df['selling_price']

In [7]:
# Feature Encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
X['model']=le.fit_transform(X['model'])

In [8]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=10)

In [9]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   model              15411 non-null  int64  
 1   vehicle_age        15411 non-null  int64  
 2   km_driven          15411 non-null  int64  
 3   seller_type        15411 non-null  object 
 4   fuel_type          15411 non-null  object 
 5   transmission_type  15411 non-null  object 
 6   mileage            15411 non-null  float64
 7   engine             15411 non-null  int64  
 8   max_power          15411 non-null  float64
 9   seats              15411 non-null  int64  
dtypes: float64(2), int64(5), object(3)
memory usage: 1.2+ MB


In [10]:
num_features=X.select_dtypes(exclude='object').columns
onehot_columns=X.select_dtypes(include='object').columns

In [11]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
oh_encodeer=OneHotEncoder(drop='first')

preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder",oh_encodeer,onehot_columns),
        ('StandardScaler',numeric_transformer,num_features)
    ],remainder='passthrough'
)

X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

# Model Training and model selection

In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [13]:
def evaluate_model(true,predicted):
    mae=mean_absolute_error(true,predicted)
    mse=mean_squared_error(true,predicted)
    rmse=np.sqrt(mse)
    r2=r2_score(true,predicted)
    print('-Mean Absolute Error: {:.4f}'.format(mae))
    print('-Root Mean Squared Error: {:.4f}'.format(rmse))
    print('-R2 score: {:.4f}\n'.format(r2))

In [14]:
models={
    'Liniear regression': LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbours Regrssor": KNeighborsRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    'AdaBoostRegressor': AdaBoostRegressor(),
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)

    # predicts
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    print(list(models.keys())[i])
    print('model performance for training data')
    evaluate_model(y_train,y_train_pred)

    print('model Performance for test data')
    evaluate_model(y_test,y_test_pred)
    print('='*35)

Liniear regression
model performance for training data
-Mean Absolute Error: 260031.9512
-Root Mean Squared Error: 478801.8476
-R2 score: 0.6763

model Performance for test data
-Mean Absolute Error: 280915.9804
-Root Mean Squared Error: 753414.8310
-R2 score: 0.5123

Lasso
model performance for training data
-Mean Absolute Error: 260030.0600
-Root Mean Squared Error: 478801.8527
-R2 score: 0.6763

model Performance for test data
-Mean Absolute Error: 280914.8828
-Root Mean Squared Error: 753414.1567
-R2 score: 0.5123

Ridge
model performance for training data
-Mean Absolute Error: 260001.6422
-Root Mean Squared Error: 478802.5439
-R2 score: 0.6763

model Performance for test data
-Mean Absolute Error: 280895.1353
-Root Mean Squared Error: 753428.2368
-R2 score: 0.5122

K-Neighbours Regrssor
model performance for training data
-Mean Absolute Error: 89544.0501
-Root Mean Squared Error: 233017.7684
-R2 score: 0.9233

model Performance for test data
-Mean Absolute Error: 119539.4097
-Root

In [15]:
knn_params={'n_neighbors':[2,4,6,8,20,30,50]}
rf_params={
    'n_estimators':[100,10,40,400],
    'max_features':[6,4,3,None,'sqrt'],
    'max_depth':[3,6,8,None],
    'min_samples_split':[100,200,500,1000]
}
ad_params={
    'n_estimators':[10,50,100,500],
    'learning_rate':[0.001,0.01,0.1,1,10],
    'loss':['linear','square','exponential'],
}
Gradient_Params={
    'learning_rate':[0.001,0.01,0.1,1,10],
    'min_samples_split':[2,10,40,70,90,110],
    'n_estimators':[10,40,90,190,400]
}

In [17]:
from sklearn.model_selection import RandomizedSearchCV
randomcv_models=[('KNN',KNeighborsRegressor(),knn_params),
                 ('RF',RandomForestRegressor(),rf_params),
                 ('AdB',AdaBoostRegressor(),ad_params),
                 ('GB',GradientBoostingRegressor(),Gradient_Params)]

In [18]:
model_param={}
for name,model,params in randomcv_models:
    random=RandomizedSearchCV(estimator=model,param_distributions=params,n_iter=100,verbose=2,cv=3,n_jobs=-1)
    random.fit(X_train,y_train)
    model_param[name]=random.best_params_

for model_name in model_param:
    print('*'*10,'Best Params for model {}'.format(model_name),'*'*10,)
    print(model_param[model_name])

C:\Users\Abrar Khan\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 7 is smaller than n_iter=100. Running 7 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 7 candidates, totalling 21 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits


C:\Users\Abrar Khan\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 60 is smaller than n_iter=100. Running 60 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 60 candidates, totalling 180 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits


C:\Users\Abrar Khan\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [ 6.25698375e-001 -4.01118428e+171  6.64527048e-001              nan
              nan -1.51015551e+076  8.03461682e-001  8.96990077e-001
  8.82453050e-001              nan  8.87718626e-001  1.36249543e-002
  3.90843738e-001  8.83168162e-001  9.12209726e-001  8.05393729e-001
  8.88775458e-001  1.36249543e-002  8.54173334e-001  6.28182518e-001
  6.73717992e-001  9.06552645e-001  9.04527585e-001  8.76608725e-001
  3.90863315e-001  9.19677132e-001  9.21376071e-001  8.86425393e-001
  6.65629265e-001  5.29263789e-002  8.91233011e-001  3.89182340e-001
  8.93928581e-001 -8.40371090e+018  5.29263789e-002  5.29263789e-002
 -8.40371090e+018  5.29263789e-002  1.11851697e-001  1.23485568e-001
  2.17980505e-001  3.91268666e-001  1.36249543e-002 -4.01118428e+171
  1.23485568e-001 -4.01118428e+171  8.96152513e-001  1.23485568e-001
  1.23485568e-001  8.75

********** Best Params for model KNN **********
{'n_neighbors': 2}
********** Best Params for model RF **********
{'n_estimators': 100, 'min_samples_split': 100, 'max_features': 6, 'max_depth': 6}
********** Best Params for model AdB **********
{'n_estimators': 50, 'loss': 'linear', 'learning_rate': 0.1}
********** Best Params for model GB **********
{'n_estimators': 400, 'min_samples_split': 2, 'learning_rate': 0.1}


In [19]:
models={
    "Random Forest Regressor":RandomForestRegressor(n_estimators=400,min_samples_split=100,max_features=6,max_depth=None),
    'K-Neighbors Regressor':KNeighborsRegressor(n_neighbors=2),
    'Ada Bosst Regressor':AdaBoostRegressor(n_estimators=500,loss='exponential',learning_rate=0.01),
    'Gradient Bosst Regressor':GradientBoostingRegressor(n_estimators=400,min_samples_split=2,learning_rate=0.1)
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)

    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    print(list(models.keys())[i])
    print("Training Data Performance")
    evaluate_model(y_train,y_train_pred)

    print("Test Data Performance")
    evaluate_model(y_test,y_test_pred)
    print('='*40,'\n')

Random Forest Regressor
Training Data Performance
-Mean Absolute Error: 116370.4028
-Root Mean Squared Error: 301809.3986
-R2 score: 0.8714

Test Data Performance
-Mean Absolute Error: 135153.2356
-Root Mean Squared Error: 649614.0489
-R2 score: 0.6374


K-Neighbors Regressor
Training Data Performance
-Mean Absolute Error: 62811.3948
-Root Mean Squared Error: 169890.2542
-R2 score: 0.9592

Test Data Performance
-Mean Absolute Error: 120853.7139
-Root Mean Squared Error: 552453.6412
-R2 score: 0.7378


Ada Bosst Regressor
Training Data Performance
-Mean Absolute Error: 224246.9637
-Root Mean Squared Error: 348414.5342
-R2 score: 0.8286

Test Data Performance
-Mean Absolute Error: 237880.1490
-Root Mean Squared Error: 648337.9226
-R2 score: 0.6388


Gradient Bosst Regressor
Training Data Performance
-Mean Absolute Error: 87227.7246
-Root Mean Squared Error: 141722.8823
-R2 score: 0.9716

Test Data Performance
-Mean Absolute Error: 111665.6736
-Root Mean Squared Error: 516169.5972
-R2 sco